# 04 - Sweep Summary Dashboard

Workhorse notebook: load one sweep's runs, identify the varying dial, and render the standard plots.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from playground.runs_io import load_sweep

In [ ]:
EXPERIMENT = "smoke_sweep"   # change to e.g. "batch_size_sharpness" once a real sweep has been fetched
runs_dir = Path.cwd().parent / "runs" / EXPERIMENT
if not runs_dir.exists():
    print(f"No runs at {runs_dir}. Run `bash scripts/sync.sh fetch` or pick another experiment.")
    df = pd.DataFrame()
else:
    df = load_sweep(runs_dir)
    print(f"loaded {len(df)} runs")
df.head()

In [ ]:
varying = []
if not df.empty:
    probe_prefixes = ("margin_", "grad_", "adversarial_", "calibration_", "thickness_", "hessian_", "landscape_")
    summary_suffixes = ("_final", "_best")
    skip_cols = {"run_dir", "experiment_name", "run_id", "extra", "description"}
    config_cols = [
        c for c in df.columns
        if not c.endswith(summary_suffixes)
        and not c.startswith(probe_prefixes)
        and c not in skip_cols
    ]
    varying = [c for c in config_cols if df[c].nunique() > 1 and c != "seed"]
    print("varying dial(s):", varying)

In [ ]:
if not df.empty and varying:
    dial = varying[0]
    fig, ax = plt.subplots(figsize=(6, 4))
    grouped = df.groupby(dial)["test_accuracy_best"].agg(["mean", "std"]).reset_index()
    ax.errorbar(grouped[dial].astype(str), grouped["mean"], yerr=grouped["std"], marker="o", capsize=4)
    ax.set_xlabel(dial)
    ax.set_ylabel("best test accuracy")
    ax.set_title(f"Test accuracy vs {dial}")
    plt.show()

In [ ]:
if not df.empty and varying and "margin_mean" in df.columns:
    dial = varying[0]
    fig, ax = plt.subplots(figsize=(6, 4))
    grouped = df.groupby(dial)["margin_mean"].agg(["mean", "std"]).reset_index()
    ax.errorbar(grouped[dial].astype(str), grouped["mean"], yerr=grouped["std"], marker="o", capsize=4)
    ax.set_xlabel(dial)
    ax.set_ylabel("mean margin")
    ax.set_title(f"Margin mean vs {dial}")
    plt.show()

In [ ]:
if not df.empty and varying and "adversarial_robust_acc_at_8_255" in df.columns:
    dial = varying[0]
    fig, ax = plt.subplots(figsize=(6, 4))
    grouped = df.groupby(dial)["adversarial_robust_acc_at_8_255"].agg(["mean", "std"]).reset_index()
    ax.errorbar(grouped[dial].astype(str), grouped["mean"], yerr=grouped["std"], marker="o", capsize=4)
    ax.set_xlabel(dial)
    ax.set_ylabel("robust accuracy @ eps = 8/255")
    ax.set_title(f"Adversarial robustness vs {dial}")
    plt.show()